# **R/S BENCHMARK — NN DATASET GENERATION (from the PCE, not the emulator)**

## Prerequisite

Run [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) then
[`02_train_pce.ipynb`](02_train_pce.ipynb) first — this notebook loads the `pce_metamodel` files
stage 2 writes, one per time step. It makes **no emulator calls and draws no latent samples**.

## Why query the PCE instead of the emulator

The per-time-step PCE (stage 2) is already fit and validated against the stochastic emulator. A
`pce_metamodel.predict(x)` call costs microseconds, against the emulator's per-point GLAM fit over
`n_latent_samples` draws. So instead of re-running the emulator to build training data for the
global NN, this notebook uses each time step's PCE as a cheap oracle: draw many fresh $(R, S)$
points, query the matching PCE, tag the result with its `Time (years)`, and stack every time step
into one dataset — the raw material [`04_train_nn.ipynb`](04_train_nn.ipynb) needs to fit
$(R, S, t) \to \lambda$ directly.

This also sidesteps the raw GLAM fit's occasional unstable `lambda 2` (a handful of design points
where the GLD least-squares fit returns an extreme value) — the PCE surface is smooth, so points
queried from it don't inherit that noise.

## 1. Libraries

In [5]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

## 2. Config

Must match [`02_train_pce.ipynb`](02_train_pce.ipynb) — `r_mean`/`r_std`/`s_mean`/`s_std` rebuild
the same `joint`, `times` must be the same grid, and `n_latent_samples` names the `pce_metamodel`
files being loaded below (it plays no role in this notebook beyond that — no latent sampling
happens here).

`n_points` is new: how many fresh $(R, S)$ points to query per time step. Since a PCE evaluation is
cheap, this can be far denser than the design sample count used to fit the PCEs themselves.

In [6]:
r_mean = 5.0
r_std  = 0.8
s_mean = 2.0
s_std  = 0.6

n_latent_samples = 2500   # must match stage 1/2 — it names the pce_metamodel files
n_lambdas        = 4
n_points         = 5000   # (R, S) query points drawn per time step for the NN dataset

times = np.linspace(0, 150, 10, endpoint=True)  # must match stage 1/2
times

array([  0.        ,  16.66666667,  33.33333333,  50.        ,
        66.66666667,  83.33333333, 100.        , 116.66666667,
       133.33333333, 150.        ])

## 3. Rebuild the joint distribution

In [7]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

## 4. Load the per-time-step PCE metamodels

One `pce_metamodel` per entry of `times`, as saved by `train_and_validate_pce_from_dataset_benchmark`
in stage 2.

In [8]:
pce_models = []
for t in times:
    with open(f'{n_latent_samples}_pce_metamodel_{t}_benchmark.pkl', 'rb') as f:
        pce_models.append(dill.load(f))

print(f"Loaded {len(pce_models)} PCE metamodels")

Loaded 10 PCE metamodels


## 5. Query the PCEs and stack the dataset

`generate_nn_dataset_benchmark` draws `n_points` fresh $(R, S)$ samples per time step, evaluates the
matching PCE on them, and stacks every time step into one dataframe with an explicit
`Time (years)` column.

In [9]:
print("="*60)
print("GENERATING THE NN DATASET FROM THE PCE MODELS")
print("="*60)

result = generate_nn_dataset_benchmark(
                                          pce_metamodels=pce_models,
                                          times=times,
                                          joint=joint,
                                          n_points=n_points,
                                          n_lambdas=n_lambdas,
                                          n_latent_samples=n_latent_samples,
                                          output_dir='.',
                                       )

df_nn = result['dataset_nn']
print(f"\nTotal rows: {len(df_nn)}")
df_nn.head()

GENERATING THE NN DATASET FROM THE PCE MODELS

----------------------------------------
GENERATING NN DATASET FROM 10 PCE MODELS
----------------------------------------
  t = 0.00 years: 5000 points queried from the PCE
  t = 16.67 years: 5000 points queried from the PCE
  t = 33.33 years: 5000 points queried from the PCE
  t = 50.00 years: 5000 points queried from the PCE
  t = 66.67 years: 5000 points queried from the PCE
  t = 83.33 years: 5000 points queried from the PCE
  t = 100.00 years: 5000 points queried from the PCE
  t = 116.67 years: 5000 points queried from the PCE
  t = 133.33 years: 5000 points queried from the PCE
  t = 150.00 years: 5000 points queried from the PCE
The NN dataset has been saved!

Total rows: 50000


,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,4.864753,2.518452,0.0,2.349110,5.249569,0.142295,0.132717
1,2.881555,1.912999,0.0,0.970985,7.498640,0.132808,0.132360
2,5.121927,1.968550,0.0,3.154849,6.114267,0.146674,0.129361
3,4.936273,1.290296,0.0,3.646073,8.077199,0.146397,0.115877
4,5.747536,2.067322,0.0,3.681521,5.652707,0.148160,0.129248


## 6. Sanity check

In [10]:
df_nn[['r', 's', 'Time (years)', 'lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']].describe()

,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,4.999780,1.998643,75.000000,0.378148,7.949196,0.134146,0.128459
std,0.798360,0.599607,47.871834,1.837130,3.338512,0.012351,0.012420
min,1.542275,-0.427960,0.000000,-4.563728,-6.670992,-0.063947,-0.045691
25%,4.460878,1.590779,33.333333,-1.101620,5.906226,0.129198,0.124596
50%,5.002311,1.998070,75.000000,0.314255,7.039106,0.136113,0.131201
75%,5.539991,2.403215,116.666667,1.775442,8.907833,0.141486,0.135815
max,8.175141,4.372668,150.000000,6.716974,59.559050,0.287202,0.271019
